<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/1_DataEngineering_%26_FinbertSentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch pandas deep-translator tqdm

import pandas as pd
import torch
from transformers import pipeline
from deep_translator import GoogleTranslator
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf


In [ ]:
df_news = pd.read_csv('/content/GoogleNews_BBCA.csv')

finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=0 if torch.cuda.is_available() else -1)
translator = GoogleTranslator(source='id', target='en')

tqdm.pandas(desc="Translating & AI Scoring")

def get_translation_and_sentiment(text):
    try:
        english_text = translator.translate(str(text)[:500])
        result = finbert(english_text)[0]
        label = result['label']

        if label == 'positive':
            score = 1
        elif label == 'negative':
            score = -1
        else:
            score = 0

        return pd.Series([english_text, score])
    except Exception:
        # Returns an empty string for translation and 0 for score on failure
        return pd.Series(["", 0])

df_news[['translated_title', 'ai_sentiment']] = df_news['title'].progress_apply(get_translation_and_sentiment)

print("\n Individual Scoring Complete! Here are 5 random headlines and their scores:")
display(df_news[['date', 'title', 'translated_title', 'ai_sentiment']].sample(5))

print("\nExporting data to CSV")
export_path = '/content/BBCA_News_Sentiment_Scored.csv'
df_news.to_csv(export_path, index=False)
print(f"Data successfully exported to: {export_path}")

In [ ]:

df_news['parsed_date'] = pd.to_datetime(df_news['date'], format='mixed', errors='coerce')
df_news['clean_date'] = df_news['parsed_date'].dt.strftime('%Y-%m-%d')

df_sentiment = df_news.groupby('clean_date')['ai_sentiment'].mean().reset_index()
df_sentiment.columns = ['Date', 'Sentiment_Score']

df_sentiment.to_csv('BBCA_Daily_Sentiment_FinBERT.csv', index=False)

print("Here is the final Average Sentiment Score per day:")
display(df_sentiment.head(5))

In [ ]:
sentiment_map = {1: 'Positive', 0: 'Neutral', -1: 'Negative'}
df_news['Sentiment_Label'] = df_news['ai_sentiment'].map(sentiment_map)

counts = df_news['Sentiment_Label'].value_counts()
percentages = df_news['Sentiment_Label'].value_counts(normalize=True) * 100
split_df = pd.DataFrame({'Count': counts, 'Percentage (%)': percentages.round(2)})

print("\n--- EXACT SENTIMENT SPLIT ---")
print(split_df)
print("-" * 30)

sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# CHART A: Pie Chart (Proportions)
# Colors: Neutral (Grey), Positive (Green), Negative (Red)
colors = ['#BDC3C7', '#2ECC71', '#E74C3C']
ax1.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140,
        colors=colors, explode=(0, 0.05, 0.05), shadow=True)
ax1.set_title('Proportion of News Sentiment (Total Headlines)', fontsize=14, fontweight='bold')

# CHART B: Bar Chart (Volume with Numbers)
sns.countplot(data=df_news, x='Sentiment_Label', palette=colors, ax=ax2, order=['Neutral', 'Positive', 'Negative'])
ax2.set_title('Volume of News Headlines per Category', fontsize=14, fontweight='bold')
ax2.set_xlabel('Sentiment Category')
ax2.set_ylabel('Number of Headlines')

# Add the exact count numbers on top of each bar
for p in ax2.patches:
    ax2.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
TICKER_SYMBOL = "BBCA.JK"
START_DATE = "2025-01-01"
END_DATE = "2026-06-07"

ticker_obj = yf.Ticker(TICKER_SYMBOL)
df_yahoo = ticker_obj.history(start=START_DATE, end=END_DATE)

if df_yahoo.empty:
    print("Critical Error: No data fetched. Please check the internet connection or ticker symbol.")
else:
    df_yahoo = df_yahoo.reset_index()

    df_yahoo['Date'] = pd.to_datetime(df_yahoo['Date']).dt.strftime('%Y-%m-%d')

    df_stock = df_yahoo[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

    for col in ['Open', 'High', 'Low', 'Close']:
        df_stock[col] = df_stock[col].round().astype(int)
    df_stock['Volume'] = df_stock['Volume'].astype(int)

    print(f"Successfully extracted {len(df_stock)} trading rows!")
    print(f"Historical span: {df_stock['Date'].min()} to {df_stock['Date'].max()}")

    df_stock.to_csv('/content/BBCA_Stock_History.csv', index=False)

In [ ]:
df_stock = pd.read_csv('/content/BBCA_Stock_History.csv')
START_DATE_BASELINE = '2025-01-01'

df_stock['Date'] = pd.to_datetime(df_stock['Date'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d')

df_stock = df_stock[df_stock['Date'] >= START_DATE_BASELINE].copy()

df_stock.rename(columns={'Price': 'Close', 'Vol.': 'Volume'}, inplace=True)

def convert_volume(vol_str):
    if pd.isna(vol_str):
        return 0
    vol_str = str(vol_str).upper().replace(',', '')
    if 'M' in vol_str:
        return float(vol_str.replace('M', '')) * 1000000
    elif 'K' in vol_str:
        return float(vol_str.replace('K', '')) * 1000
    elif 'B' in vol_str:
        return float(vol_str.replace('B', '')) * 1000000000
    else:
        return float(vol_str)

print("Converting Volume format...")
df_stock['Volume'] = df_stock['Volume'].apply(convert_volume)

# Sort chronologically
df_stock = df_stock.sort_values('Date').reset_index(drop=True)
stock_clean = df_stock[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

print("Stock Data Cleaned and Sorted!")
display(stock_clean.head(3))

In [ ]:
print("Fusing Stock Data with Daily Sentiment Scores...")

# Left join: Match all active trading days with their respective news sentiment
master_df = pd.merge(stock_clean, df_sentiment, on='Date', how='left')

# Sliding window 1 for time lag
master_df['Sentiment_Score'] = master_df['Sentiment_Score'].shift(1)

# If there were no news articles on a trading day, default the sentiment to 0 (Neutral)
master_df['Sentiment_Score'] = master_df['Sentiment_Score'].fillna(0)

# Export the final Master Dataset
output_name = 'BBCA_Master_Dataset_BiLSTM.csv'
master_df.to_csv(output_name, index=False)

print(f"File saved as: {output_name}")
print(f"Total Trading Days: {len(master_df)}")
display(master_df.head())